# Phase 5 · Coreference-Aware Span Extension

> *"`Northwind Energy Ltd` was founded in 1998. `Northwind` operates two plants in the UK. In 2019 `Northwind` acquired a smaller rival."*

Plain NER tags `Northwind Energy Ltd` on the first mention but is noticeably
patchier on the bare `Northwind` references afterwards. From a redaction
standpoint, the misses *are* leakage — the second mention of a name is no
less identifying than the first.

Phase 5 adds a deterministic post-processor — `CorefExtender` — that runs
after NER+regex. For every PERSON/ORG mention the model found, it
generates likely shorthand forms and scans the document for any matches
that NER missed. New matches inherit the entity type and identifier role
of the parent.

This notebook:

1. Walks through the candidate-generation rules (what coref forms we try).
2. Demonstrates the lift on a synthetic example.
3. Discusses the false-positive risks (`Sofia` the city vs `Sofia District Court`).
4. Points you at the TAB mention-recall evaluation script.

---


## Setup


In [ ]:
import sys
sys.path.insert(0, "../../src")

from anonymisation.pipeline import LitePipeline, extend_with_coref
from anonymisation.pipeline.coref import _candidate_forms


## What candidate forms does CorefExtender try?

The rules are conservative and TAB-specific:

**PERSON**
- Surname alone (`Maria Petrova` → `Petrova`)
- First name alone (`Maria Petrova` → `Maria`)
- Honorific + surname (`Petrova` → also try `Mr Petrova`, `Mrs Petrova`, `Ms Petrova`, `Dr Petrova`)

**ORG**
- First *significant* word (not in `_GENERIC_ORG_WORDS` like `Ltd`, `Inc`, `Holdings`, `the`, `International`, etc.)
- Two-word prefix when the first two words are both non-generic (`Sofia District Court` → `Sofia District`, **not** `Sofia` alone)

**Excluded**
- Any candidate shorter than 3 characters (avoids `Co`, `Mr`, `Inc`)
- Generic legal-doc words used standalone (`court`, `tribunal`, `commission`, etc.)
- Pronouns and articles (`he`, `she`, `the`)


In [ ]:
# Inspect the generated candidates
for surface, etype in [
    ("Maria Petrova",           "PERSON"),
    ("Sir Lawrence Smith",      "PERSON"),
    ("Dr John Smith",           "PERSON"),
    ("Northwind Energy Ltd",    "ORG"),
    ("Acme Holdings Ltd",       "ORG"),
    ("Sofia District Court",    "ORG"),
    ("MegaCorp International",  "ORG"),
    ("the European Commission", "ORG"),
]:
    forms = _candidate_forms(surface, etype)
    print(f"  {surface!r:35s} ({etype:6s}) →  {forms}")


Two things to notice:

- **`Sofia District Court` → `Sofia District`, not `Sofia` alone.** Two-word prefixes are kept (much less ambiguous), but the bare first word is dropped because `Sofia` on its own could refer to the city.
- **`the European Commission` → `European`.** Conservative — we strip the leading `the`, take the next significant word. In a real corpus this is risky (matches the adjective elsewhere), which is why coref-derived spans are marked `source="coref"` with `confidence=0.7`. Downstream consumers can filter by confidence.


## The lift on a synthetic example

NER (real or hand-crafted) typically only tags the *first* full mention of an entity. Without `CorefExtender`, the shorthand references below are missed entirely. With it enabled they get picked up — and any redaction pipeline downstream now sees them as DIRECT identifiers.


In [ ]:
text = (
    "Northwind Energy Ltd was founded in 1998. Northwind operates two plants "
    "in the UK. In 2019 Northwind acquired Stark Holdings, a smaller rival. "
    "Maria Petrova, the CEO of Northwind, announced the deal. Mrs Petrova "
    "later joined the board of Stark."
)

# Pretend NER only caught the FULL forms — typical real-world behaviour
ner_only = [
    (0,  19, "ORG",    "Northwind Energy Ltd"),
    (87, 102, "ORG",   "Stark Holdings"),
    (104, 117, "PERSON","Maria Petrova"),
]

def fixed_ner(_text):
    return ner_only

# Pipeline with coref OFF — only the explicit NER spans get caught
lite_off = LitePipeline(ner_provider=fixed_ner, coref_extend=False)
print("── COREF OFF ──")
print("  " + lite_off(text).redacted_text)
print()

# Pipeline with coref ON — CorefExtender finds the shorthand mentions
lite_on = LitePipeline(ner_provider=fixed_ner, coref_extend=True)
print("── COREF ON ──")
print("  " + lite_on(text).redacted_text)


Notice the differences:

- `Northwind` (chars 42, 71, 162) — three additional mentions now redacted.
- `Stark` (final sentence) — caught from the prior `Stark Holdings` mention.
- `Mrs Petrova` (final sentence) — caught from `Maria Petrova` via the honorific-form candidate.

Every one of these would have been a leakage in the coref-OFF version.


## False-positive risks — what can go wrong

The substring rule is fast and predictable but it isn't free. Two failure modes worth knowing:

1. **Geographic collisions.** If a document references `Sofia District Court` (ORG) and *also* mentions Sofia the capital city (LOC), we deliberately avoid generating the standalone `Sofia` form for the ORG — but if NER missed the city LOC too, the city mention will go unredacted. This is a recall miss, not a false positive — better than the alternative.

2. **Shared surnames.** Two different people named `Smith` in the same document — say a `John Smith` plaintiff and a `Jane Smith` witness — will be merged by the substring rule. The first occurrence of `Smith` in the text gets attached to whichever full mention NER found first. A proper coref model would distinguish them.

For both failure modes, the right fix is approach (2) from the writeup — drop in `fastcoref` or spaCy's `experimental_coref`. That's Phase 5.1 territory.


## Evaluating against TAB

TAB's gold annotations include an `entity_id` linking every mention of the same entity. That gives us a clean way to measure how much CorefExtender lifts recall on the shorthand mentions that pure NER misses.

The script lives at `phase5_coreference/evaluate_mention_recall.py`. Run from the repo root:

```bash
# Quick smoke test (first 50 docs, ~30 seconds)
python phase5_coreference/evaluate_mention_recall.py --sample 50

# Full run (~8 min on en_core_web_trf, CPU)
python phase5_coreference/evaluate_mention_recall.py
```

Output:

- `phase5_coreference/results/mention_recall.csv` — one row per gold entity per pipeline variant.
- `phase5_coreference/results/mention_recall_summary.json` — aggregate numbers (macro + micro recall, per-entity-type breakdown).

The expected lift is biggest on `PERSON` and `ORG` — those are the labels where (a) shorthand reference is common and (b) the substring rule is most reliable. `DATETIME`, `QUANTITY`, `DEM` benefit less because those entities rarely have a coref structure in the first place.


## What this phase does NOT do

- **No off-the-shelf coref model.** `fastcoref` or `spacy-coref` would catch pronouns and harder semantic coref. The substring rule doesn't.
- **No retraining.** This is a post-processor, not a model improvement. The model's first-mention recall is unchanged — we're only improving subsequent-mention recall via the substring + honorific heuristic.
- **No cross-document coref.** Same scope as Phase 4 — each document is processed independently.

The TAB evaluation is the right way to know whether the heuristic is worth its weight in your domain. If the lift is small (e.g. < 2 pp on PERSON recall), a real coref model is probably the better next investment. If the lift is large, you've added meaningful safety at zero training cost.
